In [1]:
import pandas as pd
import sys
from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(PROJECT_ROOT)




/home/abril/UdeSA/ML/TP_Final_ML


In [3]:
from category_utils import (
    load_dataset,
    get_categorical_columns,
    build_category_summary,
    build_category_table,
    save_category_audit_report,
)
df = load_dataset()

In [4]:
import re
import unicodedata
from difflib import SequenceMatcher

import pandas as pd


def normalize_text(value):
    """
    Normalizes text values to make category comparison easier.

    Arguments:
        value (object): original category value

    Returns:
        str: normalized text value
    """
    if pd.isna(value):
        return "missing"

    text = str(value).lower().strip()

    # Remove accents, punctuation, and duplicated spaces.
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def text_similarity(first_value, second_value):
    """
    Computes similarity between two text values.

    Arguments:
        first_value (str): first normalized text value
        second_value (str): second normalized text value

    Returns:
        float: similarity score between 0 and 1
    """
    return SequenceMatcher(None, first_value, second_value).ratio()


def suggest_category_mapping(series, threshold=0.85):
    """
    Suggests category mappings based on text similarity and category frequency.

    Arguments:
        series (pd.Series): categorical variable to analyze
        threshold (float): minimum similarity required to suggest a mapping

    Returns:
        pd.DataFrame: suggested category mapping table
    """
    counts = series.dropna().astype(str).str.strip().value_counts()
    categories = counts.index.tolist()

    rows = []

    for index, category in enumerate(categories):
        category_norm = normalize_text(category)

        best_match = category
        best_score = 1.0
        best_match_count = counts[category]
        action = "keep"

        # Only compare against more frequent categories.
        for candidate in categories[:index]:
            candidate_norm = normalize_text(candidate)
            score = text_similarity(category_norm, candidate_norm)

            if score > best_score or action == "keep" and score >= threshold:
                best_match = candidate
                best_score = score
                best_match_count = counts[candidate]
                action = "map"

        rows.append({
            "original_category": category,
            "original_count": counts[category],
            "suggested_category": best_match,
            "suggested_count": best_match_count,
            "similarity": round(best_score, 3),
            "action": action,
        })

    return pd.DataFrame(rows)


def apply_category_mapping(series, mapping_table, manual_mapping=None):
    """
    Applies a category mapping to a categorical variable.

    Arguments:
        series (pd.Series): categorical variable to transform
        mapping_table (pd.DataFrame): mapping table created by suggest_category_mapping
        manual_mapping (dict | None): manually approved or corrected mappings

    Returns:
        pd.Series: transformed categorical variable
    """
    mapping = dict(
        zip(
            mapping_table["original_category"],
            mapping_table["suggested_category"],
        )
    )

    if manual_mapping is not None:
        mapping.update(manual_mapping)

    return series.map(lambda value: mapping.get(str(value).strip(), value) if not pd.isna(value) else value)

In [8]:
color_mapping = suggest_category_mapping(df["Modelo"], threshold=0.85,)

display(color_mapping[color_mapping["action"] == "map"])

,original_category,original_count,suggested_category,suggested_count,similarity,action
44,Clase GLA,84,Clase GLC,104,0.889,map
61,Clase GLE,39,Clase GLC,104,0.889,map
65,C5 Aircross,33,C3 Aircross,228,0.909,map
68,Tiggo 2,30,Tiggo 3,104,0.857,map
69,Tiggo 5,30,Tiggo 3,104,0.857,map
71,Clase GLK,29,Clase GLC,104,0.889,map
77,q5 sportback,24,Q3 Sportback,41,0.917,map
124,Tiggo 4,2,Tiggo 3,104,0.857,map
126,208,2,2008,1144,0.857,map
128,C4 Aircross,2,C3 Aircross,228,0.909,map
